# Feature Engineering Experiments

Explore candidate features across 5 models to find the best combination for fraud detection.


## 1. Imports


In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    precision_score, recall_score, precision_recall_curve, roc_curve
)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
print("Imports done")


## 2. Load Data


In [ ]:
df = pd.read_csv("data/raw/creditcard.csv")
print(f"Shape: {df.shape}  |  Fraud rate: {df['Class'].mean()*100:.3f}%")

TARGET = "Class"
X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

ratio = (y_train == 0).sum() / (y_train == 1).sum()
print(f"Train: {X_train.shape}  Test: {X_test.shape}")
print(f"Fraud: train={y_train.mean()*100:.3f}%  test={y_test.mean()*100:.3f}%")
print(f"Neg/Pos ratio: {ratio:.1f}")


## 3. Feature Engineering

All candidate features are defined here. Different subsets will be tested later.


In [ ]:
def engineer_features(X):
    X = X.copy()

    X["Amount_log"] = np.log1p(X["Amount"])

    hour = (X["Time"] // 3600) % 24
    X["Hour_sin"] = np.sin(2 * np.pi * hour / 24)
    X["Hour_cos"] = np.cos(2 * np.pi * hour / 24)

    v_cols = [c for c in X.columns if c.startswith("V")]

    X["V_mean"] = X[v_cols].mean(axis=1)
    X["V_std"] = X[v_cols].std(axis=1)
    X["V_max"] = X[v_cols].max(axis=1)
    X["V_min"] = X[v_cols].min(axis=1)
    X["V_range"] = X["V_max"] - X["V_min"]

    X["V_outliers"] = (np.abs(X[v_cols]) > 3).sum(axis=1)
    X["V_skew"] = X[v_cols].skew(axis=1)
    X["V_mad"] = X[v_cols].sub(X[v_cols].median(axis=1), axis=0).abs().median(axis=1)
    X["V_sum_sq"] = (X[v_cols] ** 2).sum(axis=1)

    X["Amount_rank"] = X["Amount_log"].rank(pct=True)
    X["Time_gap"] = X["Time"].diff().fillna(0)

    return X, v_cols


X_train, v_cols = engineer_features(X_train)
X_test,  _      = engineer_features(X_test)

X_train.drop(columns=["Time", "Amount"], inplace=True)
X_test.drop(columns=["Time", "Amount"], inplace=True)

print(f"Total features: {X_train.shape[1]}")
print(X_train.columns.tolist())


## 4. Candidate Feature Inspection

Visual check: how well does each engineered feature separate fraud from normal?


In [ ]:
eng_cols = [c for c in X_train.columns if not c.startswith("V")]
fig, axes = plt.subplots(3, 4, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(eng_cols[:12]):
    train_df = pd.DataFrame({col: X_train[col], "Class": y_train.values})
    for label, color, name in [(0, "steelblue", "Normal"), (1, "crimson", "Fraud")]:
        subset = train_df[train_df["Class"] == label][col]
        axes[i].hist(subset, bins=80, density=True, alpha=0.5, color=color, label=name)
    axes[i].set_title(col, fontsize=10)
    axes[i].legend(fontsize=7)

for j in range(i + 1, len(axes)):
    axes[j].axis("off")

plt.suptitle("Feature Distributions: Normal vs Fraud", fontsize=13)
plt.tight_layout()
plt.show()


## 5. Feature Sets & Evaluation Helper


In [ ]:
base_eng = ["Amount_log", "Hour_sin", "Hour_cos"]
v_stats  = ["V_mean", "V_std", "V_max", "V_min"]
new_feats = [
    "V_range", "V_outliers", "V_skew", "V_mad",
    "V_sum_sq", "Amount_rank", "Time_gap"
]

SCALE_COLS = base_eng + v_stats + new_feats

feature_sets = {
    "A (V only)": v_cols,
    "B (+Amt,Time)": v_cols + base_eng,
    "C (+V-stats)": v_cols + base_eng + v_stats,
    "D (+new)": v_cols + base_eng + v_stats + new_feats,
}


def evaluate(model, Xtr, ytr, Xte, yte, scale=False):
    if scale:
        sx = StandardScaler()
        cols = [c for c in SCALE_COLS if c in Xtr.columns]
        Xtr = Xtr.copy()
        Xte = Xte.copy()
        Xtr[cols] = sx.fit_transform(Xtr[cols])
        Xte[cols] = sx.transform(Xte[cols])
    model.fit(Xtr, ytr)
    probs = model.predict_proba(Xte)[:, 1]
    preds = (probs >= 0.5).astype(int)
    return {
        "PR_AUC": average_precision_score(yte, probs),
        "ROC_AUC": roc_auc_score(yte, probs),
        "F1": f1_score(yte, preds),
        "Precision": precision_score(yte, preds, zero_division=0),
        "Recall": recall_score(yte, preds, zero_division=0),
        "probs": probs,
    }


## 6. Run Experiments

5 feature sets x 5 models = 25 combinations. Lightweight params for speed.


In [ ]:
models = {
    "LogisticRegression": LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42),
    "RandomForest": RandomForestClassifier(n_estimators=200, max_depth=10, class_weight="balanced", random_state=42, n_jobs=-1),
    "XGBoost": XGBClassifier(n_estimators=200, learning_rate=0.1, max_depth=6, scale_pos_weight=ratio, random_state=42, n_jobs=-1),
    "LightGBM": LGBMClassifier(n_estimators=200, learning_rate=0.1, max_depth=8, num_leaves=32, class_weight="balanced", random_state=42, n_jobs=-1, verbose=-1),
    "CatBoost": CatBoostClassifier(iterations=200, learning_rate=0.1, depth=6, auto_class_weights="Balanced", random_state=42, verbose=0),
}

all_results = []

for set_name, cols in feature_sets.items():
    Xtr = X_train[cols]
    Xte = X_test[cols]
    scale = set_name != "A (V only)"
    for model_name, model in models.items():
        score = evaluate(model, Xtr, y_train, Xte, y_test, scale=scale)
        all_results.append({"Set": set_name, "Model": model_name, **score})
        print(f"{set_name:15s} | {model_name:20s} | PR={score['PR_AUC']:.4f} | ROC={score['ROC_AUC']:.4f} | F1={score['F1']:.4f}")

results_df = pd.DataFrame(all_results)
pivot_pr = results_df.pivot_table(index="Set", columns="Model", values="PR_AUC", aggfunc="first")
print("\n" + "=" * 70)
print("PR-AUC by Feature Set x Model")
print(pivot_pr.round(4).to_string())


## 7. Results Analysis

Visual comparison: which feature set + model wins?


In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
x = np.arange(len(pivot_pr.index))
width = 0.15
colors = ["#4e79a7", "#f28e2b", "#e15759", "#76b7b2", "#59a14f"]
for i, model in enumerate(pivot_pr.columns):
    vals = pivot_pr[model].values
    ax.bar(x + i * width, vals, width, label=model, color=colors[i % len(colors)])
ax.set_xticks(x + width * 2)
ax.set_xticklabels(pivot_pr.index)
ax.set_ylabel("PR-AUC")
ax.set_title("PR-AUC by Feature Set and Model")
ax.legend(fontsize=8)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

print("Best set per model:")
for m in pivot_pr.columns:
    best_set = pivot_pr[m].idxmax()
    best_val = pivot_pr[m].max()
    print(f"  {m:20s} -> {best_set:15s} ({best_val:.4f})")

print("\nOverall best combo:")
best_row = results_df.loc[results_df["PR_AUC"].idxmax()]
print(f"  {best_row['Set']} + {best_row['Model']}  (PR-AUC={best_row['PR_AUC']:.4f})")


## 8. Winning Combo Deep Dive

PR curve, ROC curve, and confusion matrix for the best combination.


In [ ]:
winner_set = best_row["Set"]
winner_model_name = best_row["Model"]
winner_model = models[winner_model_name]
winner_cols = feature_sets[winner_set]

Xtr_w = X_train[winner_cols]
Xte_w = X_test[winner_cols]
scale = winner_set != "A (V only)"
if scale:
    sx = StandardScaler()
    scale_cols = [c for c in SCALE_COLS if c in Xtr_w.columns]
    Xtr_w = Xtr_w.copy()
    Xte_w = Xte_w.copy()
    Xtr_w[scale_cols] = sx.fit_transform(Xtr_w[scale_cols])
    Xte_w[scale_cols] = sx.transform(Xte_w[scale_cols])
winner_model.fit(Xtr_w, y_train)
probs_w = winner_model.predict_proba(Xte_w)[:, 1]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

prec, rec, _ = precision_recall_curve(y_test, probs_w)
axes[0].plot(rec, prec, color="crimson", lw=2.5, label=f"PR-AUC={best_row['PR_AUC']:.4f}")
axes[0].axhline(y_test.mean(), color="grey", ls="--", alpha=0.5, label="No-skill")
axes[0].set_xlabel("Recall")
axes[0].set_ylabel("Precision")
axes[0].set_title("Precision-Recall Curve")
axes[0].legend()

fpr, tpr, _ = roc_curve(y_test, probs_w)
axes[1].plot(fpr, tpr, color="steelblue", lw=2.5, label=f"ROC-AUC={best_row['ROC_AUC']:.4f}")
axes[1].plot([0, 1], [0, 1], "grey", ls="--", alpha=0.5)
axes[1].set_xlabel("FPR")
axes[1].set_ylabel("TPR")
axes[1].set_title("ROC Curve")
axes[1].legend()

preds_w = (probs_w >= 0.5).astype(int)
cm = pd.crosstab(y_test, preds_w, rownames=["Actual"], colnames=["Predicted"], margins=True)
axes[2].axis("off")
axes[2].table(cellText=cm.values, rowLabels=cm.index, colLabels=cm.columns,
              loc="center", cellLoc="center")
axes[2].set_title("Confusion Matrix @ 0.5")

plt.suptitle(f"Winning Combo: {winner_set} + {winner_model_name}", fontsize=13)
plt.tight_layout()
plt.show()


## 9. Threshold Tuning

Find the optimal decision threshold for the winning model.


In [ ]:
thresholds = np.arange(0.05, 0.95, 0.02)
f1s, precs, recs = [], [], []
for t in thresholds:
    p = (probs_w >= t).astype(int)
    f1s.append(f1_score(y_test, p, zero_division=0))
    precs.append(precision_score(y_test, p, zero_division=0))
    recs.append(recall_score(y_test, p, zero_division=0))

best_idx = np.argmax(f1s)
best_t = thresholds[best_idx]
best_f1 = f1s[best_idx]
default_f1 = f1_score(y_test, (probs_w >= 0.5).astype(int))

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(thresholds, f1s,   lw=2.5, label="F1", color="steelblue")
ax.plot(thresholds, precs, lw=2, label="Precision", color="green", alpha=0.7)
ax.plot(thresholds, recs,  lw=2, label="Recall", color="crimson", alpha=0.7)
ax.axvline(best_t, color="black", ls="--", lw=1.5, label=f"Best F1 @ t={best_t:.2f}")
ax.axvline(0.5, color="grey", ls=":", lw=1, label="Default 0.5")
ax.set_xlabel("Threshold")
ax.set_ylabel("Score")
ax.set_title(f"Threshold Tuning — {winner_set} + {winner_model_name}")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Default threshold (0.50): F1={default_f1:.4f}")
print(f"Optimal threshold ({best_t:.2f}): F1={best_f1:.4f} (delta={best_f1-default_f1:+.4f})")


## 10. Summary


In [ ]:
print("=== EXPERIMENT SUMMARY ===\n")
print(f"Best feature set  : {winner_set}")
print(f"Best model        : {winner_model_name}")
print(f"PR-AUC            : {best_row['PR_AUC']:.4f}")
print(f"ROC-AUC           : {best_row['ROC_AUC']:.4f}")
print(f"F1 (t=0.5)        : {best_row['F1']:.4f}")
print(f"Optimal F1        : {best_f1:.4f} @ threshold={best_t:.2f}")
print()

print("Feature set ranking (avg PR-AUC across models):")
for s, v in results_df.groupby("Set")["PR_AUC"].mean().sort_values(ascending=False).items():
    print(f"  {s:15s}: {v:.4f}")
print()

print("Model ranking (avg PR-AUC across sets):")
for m, v in results_df.groupby("Model")["PR_AUC"].mean().sort_values(ascending=False).items():
    print(f"  {m:20s}: {v:.4f}")
